# FuelWise AI: entrenamiento en Google Colab

Ejecuta el pipeline completo: descarga de VED, preprocesamiento, entrenamiento de LSTM y GRU, evaluación y modelo de valor. Al final descarga los artefactos para usar la app en local.

**Antes de empezar:** en `Entorno de ejecución`, elegir `T4 GPU`. La GPU es opcional. En CPU el entrenamiento tarda de 15 a 20 min y con T4 de 3 a 5 min. El preprocesamiento usa CPU y tarda de 5 a 8 min.

In [ ]:
!nvidia-smi -L || echo 'Sin GPU: se usará CPU'

## 1. Obtener el proyecto

In [ ]:
!git clone https://github.com/miafuentes30/CC3092-PRY3.git fuelwise

In [ ]:
%cd /content/fuelwise
!pip install -q py7zr streamlit plotly openpyxl pyarrow

## 2. Pipeline completo
Descarga unos 176 MB del repositorio gsoh/VED, descomprime unos 3 GB de CSV y entrena ambos modelos.

In [ ]:
!python run_all.py

## 3. Resultados

In [ ]:
import json, pandas as pd
m = json.load(open('artifacts/metrics.json'))
display(pd.DataFrame(m['block']['test']).T.round(3))
display(pd.DataFrame(m['trip']['test']).T.round(3))
v = json.load(open('artifacts/value_results.json'))
display(pd.DataFrame({k: x['test'] for k, x in v['methods'].items()}).T.round(3))
display(pd.read_csv('artifacts/iterations.csv'))

## 4. Opcional: iteración 1, modelo directo
Entrena prediciendo el consumo sin la corrección sobre la persistencia. Después vuelve a entrenar el modelo residual final.

In [ ]:
# !python -m src.train --target direct && python -m src.evaluate
# !python -m src.train --target residual && python -m src.evaluate && python -m src.value_model

## 5. Descargar artefactos
Descomprimir `artifacts.zip` en la raíz del repositorio local, reemplazando `artifacts/`.

In [ ]:
!zip -q -r artifacts.zip artifacts
from google.colab import files
files.download('artifacts.zip')

## 6. Opcional: abrir la app desde Colab
Crea un enlace temporal con Cloudflare Tunnel. Para la feria es más estable correr la app en local.

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
!nohup streamlit run app/app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
!sleep 5 && ./cloudflared tunnel --url http://localhost:8501 2>&1 | grep -m1 -o 'https://.*trycloudflare.com'